In [3]:
'''
 Agents with Custom Tools 
 The Simple FunctionTool: Calling a Real-Time Weather API
The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.
Key Concept: The function's docstring is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose,
parameters, and when to use it.
we'll create a tool that calls the free, public U.S. National Weather Service API to get a real-time forecast. No API key needed!
'''
import requests
import json

# --- Tool Definition: A function that calls a live public API ---
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.
    Args:
        location: The city name, e.g., "bengaluru".
    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

In [4]:
get_live_weather_forecast("San Francisco")

🛠️ TOOL CALLED: get_live_weather_forecast(location='San Francisco')


{'status': 'success',
 'temperature': '70°F',
 'forecast': 'Mostly sunny. High near 70, with temperatures falling to around 67 in the afternoon. West southwest wind 5 to 13 mph, with gusts as high as 18 mph.'}

In [26]:
from google.adk.agents import Agent
from google.adk.sessions import InMemorySessionService, Session
from uuid import uuid4
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part
from dotenv import load_dotenv

In [27]:
load_dotenv()

True

In [28]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [29]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [30]:
# --- Agent Definition: An agent that USES the new tool ---
weather_agent = Agent(
    name = "weather_aware_planner",
    model = "gemini-3.5-flash",
    description = "A trip planner that checks the real-time weather before making suggestions.",
    instruction = "You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools = [get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [31]:
# --- testing the Weather-Aware Planner ---
async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name = weather_agent.name,user_id = my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

 Running query for agent: 'weather_aware_planner' in session: '2b8bf47d-af75-4118-8d5b-d898c0b2fa29'...
EVENT:model_version='gemini-3.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='call_21924',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\x12\xe5\x03\n\xe2\x03\x01\x11M2\x0fbs"\r\xb8\x84\xff\x837\xbc\x06\xe6\xec\xf6\xdeA\xc7\x05;K\xcfY\xf3\x96\xe4\x81\xe2\x17#s0d\x99\x80\x7f\xc7\xb2\x83(\x9dB\xe7\xee\x81-\xact\xcb\xdf\x10\x98\xbe>\xb6\x99v\x91\xefiT[\x0f\xc0\x84{\x01u\xa5\xb4\x8fnw\xe5\xea\xd8\xbd\xafu2K\x82DY\xdabU...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentR

The weather near Lake Tahoe is currently perfect for hiking! It is sunny with a comfortable temperature of 77°F and a gentle west wind of 0 to 10 mph. 

With such clear and pleasant conditions, it is a great day to hit the trails. Here are a few hiking recommendations near Lake Tahoe:

1. **Emerald Bay State Park (Rubicon Trail):** A beautiful trail offering stunning views of the turquoise waters of Emerald Bay and Fannette Island.
2. **Mount Tallac Trail:** For a more challenging hike with rewarding panoramic views of the entire Lake Tahoe basin.
3. **Eagle Falls to Eagle Lake:** A shorter, scenic hike that features a waterfall and leads to a serene alpine lake.

Since it is sunny and 77°F, make sure to pack plenty of water, wear sunscreen, and bring a hat for sun protection. Enjoy your hike!

--------------------------------------------------

